# Eda on silver layer 
#### This notebook validates the transformations made in the silver layer so that they match what we found in the bronze EDA

In [0]:
silver = spark.read.table("marathos.silver.cleaned_marathos_2")
display(silver)

In [0]:
print(f"Number of rows: {silver.count():,}")


In [0]:
silver.printSchema()

In [0]:
# Bronze to Silver loss ratio
df_bronze = spark.table("marathos.bronze.raw_supply_chain")
df_silver = spark.table("marathos.silver.cleaned_marathos_2")

bronze_count = df_bronze.count()
silver_count = df_silver.count()

print(f"Bronze rows: {bronze_count:,}")
print(f"Silver rows: {silver_count:,}")
print(f"Dropped: {bronze_count - silver_count:,} ({(1 - silver_count/bronze_count):.1%})")

### Did XXX (unknown) athletes get dropped?
Bronze EDA flagged `XXX` as unknown nationality — should be filtered out.
Should be 0.

In [0]:
from pyspark.sql.functions import col

xxx_rows = silver.filter(col("athlete_country") == "XXX").count()
print(f"XXX country rows: {xxx_rows}")

### Did day-unit (`d`) 
Format in bronze is glued, e.g. "16d", "8d" —
no space or word boundary before the "d".

In [0]:
day_events = silver.filter(col("event_distance_length").rlike(r"(?i)\d+d$")).count()
print(f"Day-unit event rows: {day_events}")

### Did stage/multi-stage events get dropped?
Bronze EDA flagged events containing "/" as stage events.
Should be 0.

In [0]:
stage_rows = silver.filter(col("event_distance_length").contains("/")).count()
print(f"Stage event rows: {stage_rows}")
# Should be 0

### Event type + unit breakdown
Only km, mi, h should have made it through — nothing else.

In [0]:
from pyspark.sql.functions import desc

silver.groupBy("event_type").count().orderBy(desc("count")).display()

### Multi-day performance dropped?
Bronze EDA said to drop performances like "3d 02:03:00 h".
Should be 0.

In [0]:
multiday_perf = silver.filter(col("athlete_performance").rlike(r"^\d+d\s+")).count()
print(f"Multi-day performance rows: {multiday_perf}")
# Should be 0

### Year of birth scope
Bronze EDA found impossible values (e.g. 1193, 2021). Should now be bounded 1900–2010,
or null if it fell outside that range.

In [0]:
from pyspark.sql.functions import min as spark_min, max as spark_max

silver.select(
    spark_min("athlete_year_of_birth").alias("min_year"),
    spark_max("athlete_year_of_birth").alias("max_year"),
).display()

### Event date parsing check

In [0]:
null_dates = silver.filter(col("event_date").isNull()).count()
print(f"Rows with null event_date: {null_dates}")

In [0]:
silver.filter(col("event_dates").contains("-")) \
      .select("event_dates", "event_date") \
      .distinct() \
      .show(20, truncate=False)

In [0]:
cross_month_pattern = r"^\d{2}\.\d{2}\.-\d{2}\.\d{2}\.\d{4}$"

bronze_cross_month = spark.table("marathos.bronze.raw_supply_chain") \
    .filter(col("Event dates").rlike(cross_month_pattern)).count()

silver_cross_month = silver.filter(col("event_dates").rlike(cross_month_pattern)).count()

print(f"Bronze cross-month range rows: {bronze_cross_month:,}")
print(f"Silver cross-month range rows: {silver_cross_month:,}")

In [0]:
# extract year to see if there is a specific period affected by this bug or if it is spread out
from pyspark.sql.functions import col, regexp_extract, desc

cross_month_pattern = r"^\d{2}\.\d{2}\.-\d{2}\.\d{2}\.\d{4}$"

bronze = spark.table("marathos.bronze.raw_supply_chain")

cross_month_rows = bronze.filter(col("Event dates").rlike(cross_month_pattern))

cross_month_rows.withColumn(
    "year", regexp_extract(col("Event dates"), r"(\d{4})$", 1)
).groupBy("year").count().orderBy(desc("count")).display()

In [0]:
# This shows if there is a specific month that is affected, for example it would be 01 for January and 31 of december
cross_month_rows.withColumn(
    "start_month", regexp_extract(col("Event dates"), r"^\d{2}\.(\d{2})\.-", 1)
).groupBy("start_month").count().orderBy(desc("count")).display()

In [0]:
# What type of events does this apply for?
cross_month_rows.groupBy("Event distance/length").count() \
    .orderBy(desc("count")).limit(20).display()

In [0]:
from pyspark.sql.functions import col

cross_month_pattern = r"^\d{2}\.\d{2}\.-\d{2}\.\d{2}\.\d{4}$"

net_affected = bronze.filter(
    col("Event dates").rlike(cross_month_pattern) &
    ~col("`Event distance/length`").contains("/") &
    ~col("`Event distance/length`").rlike(r"(?i)\d+d$") &
    (col("Athlete country") != "XXX") &
    ~col("Athlete performance").rlike(r"^\d+d\s+")
).count()

print(f"Rows lost to date bug that would otherwise have survived: {net_affected:,}")

###### Regex drops 71 498 rows of cross-month date ranges are being dropped silently in silver. That is 

### Average speed recalculation check
Bronze EDA found athlete_average_speed stored as strings, non-numeric
values present. Silver recalculates it from performance + distance and filters it.

In [0]:
silver.select("athlete_average_speed").summary(
    "min", "25%", "50%", "75%", "max"
).show()

In [0]:
zero_or_null_speed = silver.filter(
    col("athlete_average_speed").isNull() | (col("athlete_average_speed") <= 0)
).count()
print(f"Rows with null/zero speed: {zero_or_null_speed}")

### Performance parsing check

In [0]:
speed_null_by_type = silver.groupBy("event_type").agg(
    col("event_type"),
).count()

silver.groupBy("event_type").count().display()

### Surrogate ID sanity check

In [0]:
print(f"Unique event_names:  {silver.select('event_name').distinct().count():,}")
print(f"Unique event_ids:    {silver.select('event_id').distinct().count():,}")
print(f"Unique athlete_ids:  {silver.select('athlete_id').distinct().count():,}")
print(f"Unique result_ids:   {silver.select('result_id').distinct().count():,}")
print(f"Total rows:          {silver.count():,}")

### Duplicate / collision check

In [0]:
duplicate_groups = (
    silver.groupBy("athlete_id", "event_id", "athlete_performance")
    .count()
    .filter(col("count") > 1)
    .count()
)

total = silver.count()
print(f"Duplicate groups: {duplicate_groups:,}")
print(f"Total rows:       {total:,}")

### Null audit
Compare against bronze EDA's null findings — only intentional nulls should remain

In [0]:
from pyspark.sql.functions import sum as spark_sum

null_counts = silver.select(
    [spark_sum(col(c).isNull().cast("int")).alias(c) for c in silver.columns]
).collect()[0].asDict()

[(c, n) for c, n in null_counts.items() if n > 0]

### Country code check
Bronze EDA flagged case duplicates (e.g. "SWE" vs "swe"). 

In [0]:
from pyspark.sql.functions import upper

case_variants = (
    silver.select("athlete_country")
    .distinct()
    .withColumn("upper", upper(col("athlete_country")))
    .groupBy("upper")
    .count()
    .filter(col("count") > 1)
)
case_variants.display()

In [0]:
mismatches = spark.sql("""
    SELECT athlete_age_category, athlete_gender, COUNT(*) AS cnt
    FROM marathos.silver.cleaned_marathos_2
    WHERE (athlete_age_category LIKE 'F%' AND athlete_gender = 'M')
       OR (athlete_age_category LIKE 'M%' AND athlete_gender = 'F')
    GROUP BY athlete_age_category, athlete_gender
    ORDER BY cnt DESC
""")

mismatch_count = mismatches.count()
print(f"Mismatched rows remaining: {mismatch_count}")
mismatches.display()

### Silver layer summary 

Bronze row count: 7 461 195
Silver layer count: 7 163 554
Rows dropped: 297 641 (4.0%)

### Issues found 

** Cross-month date ranges silently dropped ** About 1% of the bronze data contain cross-month dates these are dropped due to regex bug. Considerring the smal amount compared to the data set this will only be fixed if everything else is finished.

### Checks that passed without issue 

- Stage races (event names/distance fields containing "/") correctly filtered, 0 remaining.
- Multi-day performances (e.g. "3d 02:03:00 h") correctly filtered, 0 remaining. 
- year-of-birth correctly bounded to 1900-2010.
- Recalculated average speed is internally consistent with performance and distance (spotchecked manually) and has no null or zero vlaues post filter.
- Performance parasing is type consistent: every distnace type row has 'event_distance_km' populated, every time type has its null, and vice versa for the underlying seconds/km helper columns.
- Surrogate IDs behave as expected: 'result_id' is unique per row with zero duplicate groups on athelete_id, event_id athlete_performance.
- Country codes are cleanly normalized to uppercase with no case variant duplicates. All XXX values are also dropped.
- Remaining nulls are all exmpected and intentional. 
- Speed cap are filterd at 0.5-20 km/h. 
- Day unit events filterd correctly and 0 day-unit rows remaining. 
- No more missmatches between the genders

### Conclussion 
The silver layer table reflects the cleaning rules defined in the bronze EDA, with one implementation bug identified during validation. That bug has been accepted sins the data loss is no more than 1% aproximetly 55k rows spread out. Which means it does not effect the data to much to be a acute problem. There for the table is considered ready for the gold-layer. 